#### Dependency Management

In [1]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime

pd.options.display.float_format = '{:,.2f}'.format

#### Defining the Origin Scope

In [2]:
# Strategic Origin Clusters
COFFEE_ORIGINS = {
    "Jimma": {"lat": 7.67, "lon": 36.83, "altitude": 1750},
    "Sidama": {"lat": 6.70, "lon": 38.42, "altitude": 1920},
    "Yirgacheffe": {"lat": 6.16, "lon": 38.20, "altitude": 1880}
}

print(f"Targeting {len(COFFEE_ORIGINS)} key East-African coffee origins.")

Targeting 3 key East-African coffee origins.


#### The NASA POWER API Engine

In [3]:
def fetch_agri_climate_data(lat, lon, start="20220101", end="20241231"):
    """
    Fetches Rainfall (PRECTOTCORR) and Temperature (T2M) from NASA POWER.
    """
    base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "PRECTOTCORR,T2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON"
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None

# Execution: Pulling data for the master dataset
weather_data_store = {}
for region, coords in COFFEE_ORIGINS.items():
    print(f"Fetching weather for {region}...")
    weather_data_store[region] = fetch_agri_climate_data(coords['lat'], coords['lon'])
    time.sleep(1)

Fetching weather for Jimma...
Fetching weather for Sidama...
Fetching weather for Yirgacheffe...


#### Building the "ECX Ground Truth" Ledger

In [7]:
def generate_ledger(weather_dict):
    master_records = []
    grades = ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5']
    grade_weights = [0.15, 0.25, 0.35, 0.20, 0.05]

    for region, w_df in weather_dict.items():
        # Aggregate to monthly to simulate harvest lots
        monthly = w_df.resample('M').agg({'PRECTOTCORR': 'sum', 'T2M': 'mean'})
        
        for date, row in monthly.iterrows():
            # Ethiopia Harvest/Arrival Window: October to April
            if date.month in [10, 11, 12, 1, 2, 3, 4]:
                num_lots = np.random.randint(5, 15)
                for _ in range(num_lots):
                    # Biological logic: Yield is a function of rainfall (bell curve)
                    # Optimal rainfall approx 100-150mm/month
                    rain_factor = np.exp(-((row['PRECTOTCORR'] - 125)**2) / (2 * 50**2))
                    base_vol = np.random.uniform(5, 25) 
                    actual_vol = base_vol * (0.5 + rain_factor)
                    
                    # Price logic: Farmgate is ~65% of Export Price
                    export_price = 4.20 + np.random.normal(0, 0.3)
                    
                    master_records.append({
                        "arrival_date": date + pd.Timedelta(days=np.random.randint(0, 28)),
                        "region": region,
                        "altitude": COFFEE_ORIGINS[region]['altitude'],
                        "grade": np.random.choice(grades, p=grade_weights),
                        "volume_mt": round(actual_vol, 2),
                        "export_price_usd_lb": round(export_price, 2),
                        "farmgate_price_usd_lb": round(export_price * 0.65, 2),
                        "cum_rainfall_mm": round(row['PRECTOTCORR'], 2),
                        "avg_temp_c": round(row['T2M'], 2)
                    })
                    
    return pd.DataFrame(master_records)

df_final = generate_ledger(weather_data_store)
df_final.to_csv("../data/raw/coffee_production_raw.csv", index=False)
print(f"Master Dataset Created: {len(df_final)} rows.")

C:\Users\tohiba\AppData\Local\Temp\ipykernel_10372\1626486390.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = w_df.resample('M').agg({'PRECTOTCORR': 'sum', 'T2M': 'mean'})


Master Dataset Created: 589 rows.


In [6]:
df_final.head()

,arrival_date,region,altitude,grade,volume_mt,export_price_usd_lb,farmgate_price_usd_lb,cum_rainfall_mm,avg_temp_c
0,2022-02-02,Jimma,1750,Grade 5,5.92,4.61,3.00,46.53,18.49
1,2022-02-21,Jimma,1750,Grade 5,6.16,4.31,2.80,46.53,18.49
2,2022-02-02,Jimma,1750,Grade 1,15.86,4.32,2.81,46.53,18.49
3,2022-02-21,Jimma,1750,Grade 4,18.52,4.25,2.76,46.53,18.49
4,2022-02-16,Jimma,1750,Grade 3,18.84,4.15,2.70,46.53,18.49
